# XQuality — Checkpoint-to-Output LLM Ablation

This notebook evaluates the following question:

> Given a saved NeoOLAF checkpoint after Layer X, can a direct LLM finalizer produce the final KG / ontology / triple JSON as well as the native NeoOLAF final pipeline?

It runs **all discovered layer checkpoints**, uses the same Python environment as NeoOLAF, calls OpenRouter through LiteLLM, parallelizes LLM calls inside each layer, and evaluates the resulting triples with the same gold precision/recall/F1 style.

Outputs are saved incrementally so the notebook can be interrupted and resumed.

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:

# ============================================================
# 0. Imports
# ============================================================
import os
import sys
import re
import json
import math
import time
import hashlib
import traceback
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

print("Python:", sys.executable)
print("tqdm available:", tqdm is not None)
print("pandas:", pd.__version__)

c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: c:\Users\henri\Documents\git\post-doc\neoolafvenv\Scripts\python.exe
tqdm available: True
pandas: 3.0.1


In [4]:

# ============================================================
# 1. Configuration
# ============================================================
# Model used for the direct finalizer ablation.
MODEL = "openrouter/openai/gpt-oss-20b"
TEMPERATURE = 0.0
MAX_TOKENS = 1800

# Parallelism. This controls parallel OpenRouter calls within each layer.
MAX_WORKERS = 4

# If True, also runs several layers in parallel. Usually keep False because each layer already parallelizes calls.
PARALLELIZE_LAYERS = False
LAYER_MAX_WORKERS = 2

# Resume / debug controls.
FORCE_RERUN = False
MAX_UNITS_PER_LAYER = None  # set to 2 or 3 for smoke test
ONLY_FIRST_N_LAYERS = None  # set to 1 for smoke test
USE_TQDM = True
VERBOSE = True
PROGRESS_EVERY_N_CALLS = 10

# Evidence formatting controls.
MAX_CHARS_PER_UNIT = 9000
MAX_ITEMS_PER_UNIT = 35
MAX_CHARS_PER_ITEM = 1200

# API retry behavior.
LLM_RETRIES = 2
RETRY_SLEEP_SECONDS = 2.0

# Predicate schema for XQuality.
ALLOWED_PREDICATES = ["TRIGGERS", "CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"]

print("MODEL =", MODEL)
print("MAX_WORKERS =", MAX_WORKERS)
print("FORCE_RERUN =", FORCE_RERUN)
print("MAX_UNITS_PER_LAYER =", MAX_UNITS_PER_LAYER)

MODEL = openrouter/openai/gpt-oss-20b
MAX_WORKERS = 4
FORCE_RERUN = False
MAX_UNITS_PER_LAYER = None


In [5]:

# ============================================================
# 2. Locate the NeoOLAF repository root and expected paths
# ============================================================
def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src" / "neoolaf").exists() and (p / "examples").exists():
            return p
    # Common fallback for running from examples/XQualityMachine32
    for p in candidates:
        if p.name.lower() == "neoolaf":
            return p
    return start

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

if load_dotenv is not None:
    load_dotenv(ROOT / ".env")

DATA_ROOT = ROOT / "data" / "XQuality"
GOLD_JSON = DATA_ROOT / "Examples" / "XQuality_all_triplets_flat_en.json"
GOLD_JSON_FR = DATA_ROOT / "Examples" / "XQuality_all_triplets_flat.json"
MACHINE32_GOLD_JSON = DATA_ROOT / "Machine32" / "machine32_triples.json"
GOLD_EXCEL = DATA_ROOT / "Machine32" / "machine32_triples.xlsx"

RUNS_ROOT = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32"
OUTPUT_ROOT = RUNS_ROOT / "checkpoint_to_output_llm_ablation"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("RUNS_ROOT =", RUNS_ROOT, "exists=", RUNS_ROOT.exists())
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("GOLD_JSON =", GOLD_JSON, "exists=", GOLD_JSON.exists())
print("MACHINE32_GOLD_JSON =", MACHINE32_GOLD_JSON, "exists=", MACHINE32_GOLD_JSON.exists())
print("GOLD_EXCEL =", GOLD_EXCEL, "exists=", GOLD_EXCEL.exists())
print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))

ROOT = C:\Users\henri\Documents\git\post-doc\NeoOLAF
RUNS_ROOT = C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32 exists= True
OUTPUT_ROOT = C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\checkpoint_to_output_llm_ablation
GOLD_JSON = C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json exists= True
MACHINE32_GOLD_JSON = C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Machine32\machine32_triples.json exists= True
GOLD_EXCEL = C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Machine32\machine32_triples.xlsx exists= True
OPENROUTER_API_KEY set: True


In [6]:

# ============================================================
# 3. Robust gold loader
# ============================================================
def _is_empty(x):
    if x is None:
        return True
    try:
        if isinstance(x, float) and pd.isna(x):
            return True
    except Exception:
        pass
    s = str(x).strip()
    return s == "" or s.lower() in {"nan", "none", "null"}


def _clean_label(x):
    if _is_empty(x):
        return ""
    if isinstance(x, dict):
        for k in ["label", "name", "text", "value", "id", "uri"]:
            if k in x and not _is_empty(x[k]):
                return _clean_label(x[k])
        return ""
    if isinstance(x, list):
        return "; ".join(_clean_label(v) for v in x if not _is_empty(v))
    return re.sub(r"\s+", " ", str(x).strip())


def _norm_key(k):
    return re.sub(r"[^a-z0-9]+", "", str(k).lower())


def _find_col(columns, candidates):
    norm_to_original = {_norm_key(c): c for c in columns}
    for cand in candidates:
        nc = _norm_key(cand)
        for nk, original in norm_to_original.items():
            if nc == nk or nc in nk or nk in nc:
                return original
    return None


def _canonical_predicate(x):
    if _is_empty(x):
        return ""
    s = str(x).strip().upper()
    s_norm = re.sub(r"[^A-Z0-9]+", "_", s).strip("_")
    aliases = {
        "TRIGGER": "TRIGGERS", "TRIGGERS": "TRIGGERS", "TRIGGERED_BY": "TRIGGERS",
        "CAUSE": "CAUSES", "CAUSES": "CAUSES", "EFFECT": "CAUSES", "EFFET": "CAUSES",
        "REQUIRES": "REQUIRES", "REQUIRE": "REQUIRES", "INTERVENTION": "REQUIRES", "ACTION": "REQUIRES",
        "HANDLED_BY": "HANDLED_BY", "RESPONSIBLE": "HANDLED_BY", "RESPONSABLE": "HANDLED_BY",
        "CHARGE_INTERVENTION": "HANDLED_BY", "CHARGE_DE_L_INTERVENTION": "HANDLED_BY",
        "REFERENCES": "REFERENCES", "REFERENCE": "REFERENCES", "REF": "REFERENCES",
    }
    return aliases.get(s_norm, s_norm)


def _dedup_triples(triples):
    seen = set()
    out = []
    for t in triples:
        subj = _clean_label(t.get("subject_label") or t.get("subject") or t.get("s") or t.get("source") or t.get("head"))
        pred = _canonical_predicate(t.get("predicate") or t.get("relation") or t.get("p") or t.get("property"))
        obj = _clean_label(t.get("object_label") or t.get("object") or t.get("o") or t.get("target") or t.get("tail"))
        if not subj or not pred or not obj:
            continue
        if pred not in ALLOWED_PREDICATES:
            continue
        key = (_normalize_label_for_match(subj), pred, _normalize_label_for_match(obj))
        if key in seen:
            continue
        seen.add(key)
        out.append({
            "triple_id": f"gold_{len(out):05d}",
            "subject_label": subj,
            "predicate": pred,
            "object_label": obj,
        })
    return out


def _flatten_json_records(obj):
    records = []
    if isinstance(obj, dict):
        keys = {_norm_key(k): k for k in obj.keys()}
        has_subj = any(k in keys for k in ["subject", "subjectlabel", "subj", "source", "head"])
        has_pred = any(k in keys for k in ["predicate", "relation", "pred", "property"])
        has_obj = any(k in keys for k in ["object", "objectlabel", "obj", "target", "tail"])
        if has_subj and has_pred and has_obj:
            records.append(obj)
        for v in obj.values():
            records.extend(_flatten_json_records(v))
    elif isinstance(obj, list):
        for item in obj:
            records.extend(_flatten_json_records(item))
    return records


def _triples_from_flat_json(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    records = _flatten_json_records(data)
    triples = []
    for r in records:
        columns = list(r.keys())
        subj_col = _find_col(columns, ["subject_label", "subject", "subj", "source", "head"])
        pred_col = _find_col(columns, ["predicate", "relation", "pred", "property"])
        obj_col = _find_col(columns, ["object_label", "object", "obj", "target", "tail"])
        if subj_col and pred_col and obj_col:
            triples.append({
                "subject_label": r.get(subj_col),
                "predicate": r.get(pred_col),
                "object_label": r.get(obj_col),
            })
    return triples


def _triples_from_flat_dataframe(df):
    triples = []
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    cols = list(df.columns)
    subj_col = _find_col(cols, ["subject_label", "subject", "subj", "source", "head"])
    pred_col = _find_col(cols, ["predicate", "relation", "pred", "property"])
    obj_col = _find_col(cols, ["object_label", "object", "obj", "target", "tail"])
    if not (subj_col and pred_col and obj_col):
        return []
    for _, row in df.iterrows():
        triples.append({
            "subject_label": row.get(subj_col),
            "predicate": row.get(pred_col),
            "object_label": row.get(obj_col),
        })
    return triples


def _triples_from_structured_dataframe(df):
    triples = []
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    cols = list(df.columns)
    alarm_col = _find_col(cols, ["alarm", "message", "texte", "text", "label", "libelle", "libellé", "alarme", "nom", "title"])
    cause_col = _find_col(cols, ["cause", "trigger", "triggers", "déclenche", "declenche"])
    effect_col = _find_col(cols, ["effect", "effet", "consequence", "conséquence"])
    intervention_col = _find_col(cols, ["intervention", "action", "requires", "requirement", "solution", "remedy", "correction"])
    responsible_col = _find_col(cols, ["responsible", "responsable", "actor", "operator", "opérateur", "charge", "chargé", "intervenant"])
    reference_col = _find_col(cols, ["reference", "référence", "ref", "page", "input", "schema", "electrical"])
    for _, row in df.iterrows():
        alarm = _clean_label(row.get(alarm_col)) if alarm_col else ""
        if not alarm:
            continue
        if cause_col:
            cause = _clean_label(row.get(cause_col))
            if cause:
                triples.append({"subject_label": cause, "predicate": "TRIGGERS", "object_label": alarm})
        if effect_col:
            effect = _clean_label(row.get(effect_col))
            if effect:
                triples.append({"subject_label": alarm, "predicate": "CAUSES", "object_label": effect})
        if intervention_col:
            intervention = _clean_label(row.get(intervention_col))
            if intervention:
                triples.append({"subject_label": alarm, "predicate": "REQUIRES", "object_label": intervention})
        if responsible_col:
            responsible = _clean_label(row.get(responsible_col))
            if responsible:
                actors = re.split(r"\s*(?:;|/|\bet\b|\band\b)\s*", responsible)
                for actor in actors:
                    actor = _clean_label(actor)
                    if actor:
                        triples.append({"subject_label": alarm, "predicate": "HANDLED_BY", "object_label": actor})
        if reference_col:
            ref = _clean_label(row.get(reference_col))
            if ref:
                triples.append({"subject_label": alarm, "predicate": "REFERENCES", "object_label": ref})
    return triples


def _triples_from_excel(path):
    triples = []
    xls = pd.ExcelFile(path)
    print(f"Excel sheets in {path.name}: {xls.sheet_names}")
    for sheet_name in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet_name)
        print(f"  Sheet {sheet_name}: shape={df.shape}; columns={list(df.columns)}")
        triples.extend(_triples_from_flat_dataframe(df))
        triples.extend(_triples_from_structured_dataframe(df))
    return triples


def load_gold_triples():
    triples = []
    json_candidates = [GOLD_JSON, MACHINE32_GOLD_JSON, GOLD_JSON_FR]
    excel_candidates = [GOLD_EXCEL]
    print("Checking gold JSON candidates:")
    for path in json_candidates:
        path = Path(path)
        print(f" - {path} | exists={path.exists()}")
        if path.exists():
            try:
                loaded = _triples_from_flat_json(path)
                print(f"   loaded JSON triples: {len(loaded)}")
                triples.extend(loaded)
            except Exception as e:
                print(f"   JSON load failed: {repr(e)}")
    print("\nChecking gold Excel candidates:")
    for path in excel_candidates:
        path = Path(path)
        print(f" - {path} | exists={path.exists()}")
        if path.exists():
            try:
                loaded = _triples_from_excel(path)
                print(f"   loaded Excel triples: {len(loaded)}")
                triples.extend(loaded)
            except Exception as e:
                print(f"   Excel load failed: {repr(e)}")
    triples = _dedup_triples(triples)
    if not triples:
        raise RuntimeError("Could not load gold triples. Files exist, but their schema was not recognized.")
    return triples

# Matching normalization is defined here because the gold loader uses it.
def _normalize_label_for_match(x):
    x = _clean_label(x).lower()
    x = x.replace("—", "-").replace("–", "-")
    x = re.sub(r"\bthe\b|\ba\b|\ban\b", " ", x)
    x = re.sub(r"[^a-z0-9àâäéèêëïîôöùûüçñ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

# Diagnostic: inspect what Jupyter sees.
print("ROOT =", ROOT)
for p in [GOLD_JSON, MACHINE32_GOLD_JSON, GOLD_EXCEL]:
    print(p, "exists=", p.exists())

gold_triples = load_gold_triples()
print("\nGold triples:", len(gold_triples))
print("Gold predicates:", Counter(t["predicate"] for t in gold_triples))
pd.DataFrame(gold_triples).head(10)

ROOT = C:\Users\henri\Documents\git\post-doc\NeoOLAF
C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json exists= True
C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Machine32\machine32_triples.json exists= True
C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Machine32\machine32_triples.xlsx exists= True
Checking gold JSON candidates:
 - C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json | exists=True
   loaded JSON triples: 0
 - C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Machine32\machine32_triples.json | exists=True
   loaded JSON triples: 0
 - C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat.json | exists=True
   loaded JSON triples: 0

Checking gold Excel candidates:
 - C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Machine32\machine32_triples.xlsx | exists=True
Excel sheets in m

,triple_id,subject_label,predicate,object_label
0,gold_00000,1,TRIGGERS,1
1,gold_00001,1,CAUSES,1
2,gold_00002,1,REQUIRES,1
3,gold_00003,1,HANDLED_BY,1
4,gold_00004,1,REFERENCES,1
5,gold_00005,2,TRIGGERS,2
6,gold_00006,2,CAUSES,2
7,gold_00007,2,REQUIRES,2
8,gold_00008,2,HANDLED_BY,2
9,gold_00009,2,REFERENCES,2


In [7]:

# ============================================================
# 4. Evaluation functions
# ============================================================
def canonical_triple(t):
    subj = _normalize_label_for_match(t.get("subject_label") or t.get("subject") or t.get("source") or t.get("head"))
    pred = _canonical_predicate(t.get("predicate") or t.get("relation") or t.get("property"))
    obj = _normalize_label_for_match(t.get("object_label") or t.get("object") or t.get("target") or t.get("tail"))
    return (subj, pred, obj)


def entity_set_from_triples(triples):
    ents = set()
    for t in triples:
        s, p, o = canonical_triple(t)
        if s:
            ents.add(s)
        if o:
            ents.add(o)
    return ents


def prf(tp, pred_total, gold_total):
    precision = tp / pred_total if pred_total else 0.0
    recall = tp / gold_total if gold_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "pred_count": pred_total, "gold_count": gold_total}


def evaluate_triples(pred_triples, gold_triples):
    pred_triples = _dedup_triples(pred_triples)
    gold_triples = _dedup_triples(gold_triples)
    pred_set = {canonical_triple(t) for t in pred_triples}
    gold_set = {canonical_triple(t) for t in gold_triples}
    pred_set = {x for x in pred_set if x[0] and x[1] and x[2]}
    gold_set = {x for x in gold_set if x[0] and x[1] and x[2]}
    rel_tp = len(pred_set & gold_set)
    relation_metrics = prf(rel_tp, len(pred_set), len(gold_set))

    pred_entities = entity_set_from_triples(pred_triples)
    gold_entities = entity_set_from_triples(gold_triples)
    ent_tp = len(pred_entities & gold_entities)
    entity_metrics = prf(ent_tp, len(pred_entities), len(gold_entities))

    per_relation = {}
    predicates = sorted(set([x[1] for x in pred_set] + [x[1] for x in gold_set]))
    for pred in predicates:
        ps = {x for x in pred_set if x[1] == pred}
        gs = {x for x in gold_set if x[1] == pred}
        per_relation[pred] = prf(len(ps & gs), len(ps), len(gs))
    return {
        "entity": entity_metrics,
        "relation": relation_metrics,
        "per_relation": per_relation,
        "counts": {
            "pred_triples": len(pred_set),
            "gold_triples": len(gold_set),
            "pred_entities": len(pred_entities),
            "gold_entities": len(gold_entities),
        }
    }

# Quick check gold self-evaluation.
print(evaluate_triples(gold_triples, gold_triples)["relation"])

{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'tp': 2195, 'pred_count': 2195, 'gold_count': 2195}


In [8]:

# ============================================================
# 5. Discover saved layer checkpoints
# ============================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _safe_get_layer_name(state, path):
    for k in ["layer", "layer_name", "name", "stage"]:
        if isinstance(state, dict) and state.get(k):
            return str(state[k])
    return path.parent.name


def discover_state_files(runs_root=RUNS_ROOT):
    rows = []
    if not runs_root.exists():
        raise RuntimeError(f"RUNS_ROOT not found: {runs_root}")
    for path in sorted(runs_root.rglob("state.json")):
        try:
            state = load_json(path)
            layer_name = _safe_get_layer_name(state, path)
            counts = state.get("counts") if isinstance(state, dict) else {}
            prompt_count = state.get("prompt_call_count") if isinstance(state, dict) else None
            rows.append({
                "state_path": path,
                "run_dir": path.parent,
                "layer_name": layer_name,
                "mtime": path.stat().st_mtime,
                "candidate_triples": counts.get("candidate_triples") if isinstance(counts, dict) else None,
                "validation_issues": counts.get("validation_issues") if isinstance(counts, dict) else None,
                "prompt_call_count": prompt_count,
            })
        except Exception as e:
            rows.append({"state_path": path, "run_dir": path.parent, "layer_name": path.parent.name, "error": repr(e), "mtime": path.stat().st_mtime})
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No state.json files found under {runs_root}")
    # Keep a stable order: path order, but put likely layer order first if names include layer number.
    def layer_num(name):
        m = re.search(r"layer\s*0*(\d+)|layer0*(\d+)|l(\d+)\b", str(name).lower())
        if not m:
            return 999
        for g in m.groups():
            if g is not None:
                return int(g)
        return 999
    df["layer_num"] = df["layer_name"].map(layer_num)
    df = df.sort_values(["layer_num", "state_path"]).reset_index(drop=True)
    if ONLY_FIRST_N_LAYERS is not None:
        df = df.head(ONLY_FIRST_N_LAYERS).copy()
    return df

states_df = discover_state_files()
print("Discovered checkpoints:", len(states_df))
display(states_df[["layer_num", "layer_name", "state_path", "candidate_triples", "validation_issues", "prompt_call_count"]].head(50))

Discovered checkpoints: 15


,layer_num,layer_name,state_path,candidate_triples,validation_issues,prompt_call_count
0,0,layer00_preprocessing,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None
1,1,layer01_linguistic_expression_extraction,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None
2,2,layer02_candidate_enrichment,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None
3,2,layer02_candidate_enrichment,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None
4,3,layer03_candidate_typing_resolution,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None
5,3,layer03_candidate_typing_resolution,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None
6,4,layer04_candidate_relation_extraction,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None
7,5,layer05_candidate_triple_generation,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None
8,6,layer06_concept_relation_induction,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None
9,7,layer07_hierarchisation,C:\Users\henri\Documents\git\post-doc\NeoOLAF\...,None,None,None


In [9]:

# ============================================================
# 6. Optional smoke-test controls
# ============================================================
# Uncomment this block for a very quick test:
# MAX_UNITS_PER_LAYER = 2
# FORCE_RERUN = True
# MAX_WORKERS = 2
# states_df = states_df.head(1).copy()
# display(states_df)

print("Layers selected:", len(states_df))
print("MAX_UNITS_PER_LAYER:", MAX_UNITS_PER_LAYER)
print("MAX_WORKERS:", MAX_WORKERS)

Layers selected: 15
MAX_UNITS_PER_LAYER: None
MAX_WORKERS: 4


In [10]:

# ============================================================
# 7. Extract state items, evidence units, and native NeoOLAF triples
# ============================================================
def truncate_text(x, max_chars=MAX_CHARS_PER_ITEM):
    s = _clean_label(x)
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + " ...[truncated]"


def compact_obj(obj, max_chars=MAX_CHARS_PER_ITEM):
    if isinstance(obj, dict):
        keep = {}
        useful_keys = [
            "id", "chunk_id", "record_id", "page", "section", "type", "label", "name", "text", "content",
            "source_text", "chunk_text", "raw_text", "translated_text", "subject_label", "predicate", "object_label",
            "subject", "object", "relation", "confidence", "provenance", "source", "target", "cause", "effect",
            "intervention", "responsible", "reference", "input", "alarm", "message", "category", "domain", "range"
        ]
        for k, v in obj.items():
            if k in useful_keys or any(token in str(k).lower() for token in ["text", "label", "chunk", "page", "subject", "object", "predicate", "relation", "cause", "effect", "intervention", "responsible", "reference"]):
                if isinstance(v, (str, int, float, bool)) or v is None:
                    keep[k] = truncate_text(v, max_chars)
                elif isinstance(v, list):
                    keep[k] = [compact_obj(x, max_chars=max_chars//2) for x in v[:5]]
                elif isinstance(v, dict):
                    keep[k] = compact_obj(v, max_chars=max_chars//2)
        return keep
    if isinstance(obj, list):
        return [compact_obj(x, max_chars=max_chars) for x in obj[:10]]
    return truncate_text(obj, max_chars)


def recursive_find_triple_like(obj, out=None):
    if out is None:
        out = []
    if isinstance(obj, dict):
        keys = {_norm_key(k): k for k in obj.keys()}
        has_s = any(k in keys for k in ["subject", "subjectlabel", "source", "head"])
        has_p = any(k in keys for k in ["predicate", "relation", "property"])
        has_o = any(k in keys for k in ["object", "objectlabel", "target", "tail"])
        if has_s and has_p and has_o:
            columns = list(obj.keys())
            s_col = _find_col(columns, ["subject_label", "subject", "source", "head"])
            p_col = _find_col(columns, ["predicate", "relation", "property"])
            o_col = _find_col(columns, ["object_label", "object", "target", "tail"])
            out.append({"subject_label": obj.get(s_col), "predicate": obj.get(p_col), "object_label": obj.get(o_col)})
        for v in obj.values():
            recursive_find_triple_like(v, out)
    elif isinstance(obj, list):
        for x in obj:
            recursive_find_triple_like(x, out)
    return out


def extract_native_triples_from_state(state):
    triples = []
    if isinstance(state, dict):
        # Prefer explicit high-value containers first.
        for key in ["candidate_triples", "reasoning_inferred_triples", "kg_triples", "triples", "relations", "candidate_relation_assertions"]:
            value = state.get(key)
            if value:
                triples.extend(recursive_find_triple_like(value))
        if not triples:
            triples.extend(recursive_find_triple_like(state))
    return _dedup_triples(triples)


def find_chunk_id(obj):
    if isinstance(obj, dict):
        for k in ["chunk_id", "record_id", "id"]:
            if obj.get(k):
                return str(obj.get(k))
        prov = obj.get("provenance")
        if isinstance(prov, dict):
            for k in ["chunk_id", "record_id", "source_id"]:
                if prov.get(k):
                    return str(prov.get(k))
    return None


def collect_candidate_items(obj, path="root", out=None, max_items=5000):
    if out is None:
        out = []
    if len(out) >= max_items:
        return out
    if isinstance(obj, dict):
        # A meaningful item has text/label/relation/triple-ish keys.
        keys_norm = {_norm_key(k) for k in obj.keys()}
        meaningful = bool(keys_norm & {
            "text", "content", "sourcetext", "chunktext", "rawtext", "translatedtext", "label", "name",
            "subjectlabel", "objectlabel", "predicate", "relation", "cause", "effect", "intervention", "responsible", "reference"
        })
        if meaningful:
            out.append({"path": path, "chunk_id": find_chunk_id(obj), "item": compact_obj(obj)})
        # Traverse likely containers.
        for k, v in obj.items():
            if isinstance(v, (list, dict)):
                collect_candidate_items(v, path=f"{path}.{k}", out=out, max_items=max_items)
    elif isinstance(obj, list):
        for i, x in enumerate(obj[:max_items]):
            collect_candidate_items(x, path=f"{path}[{i}]", out=out, max_items=max_items)
    return out


def build_evidence_units(state, layer_name):
    items = collect_candidate_items(state, max_items=8000)
    # Group by chunk_id when available, otherwise create batches.
    by_chunk = defaultdict(list)
    no_chunk = []
    for it in items:
        cid = it.get("chunk_id")
        if cid:
            by_chunk[cid].append(it)
        else:
            no_chunk.append(it)

    units = []
    for cid, chunk_items in by_chunk.items():
        chunk_items = chunk_items[:MAX_ITEMS_PER_UNIT]
        payload = {
            "layer_name": layer_name,
            "chunk_id": cid,
            "items": chunk_items,
        }
        text = json.dumps(payload, ensure_ascii=False, indent=2)
        if len(text) > MAX_CHARS_PER_UNIT:
            text = text[:MAX_CHARS_PER_UNIT] + "\n...[unit truncated]"
        units.append({"unit_id": f"{layer_name}__{cid}", "chunk_id": cid, "text": text})

    # Batch no-chunk items.
    batch_size = MAX_ITEMS_PER_UNIT
    for start in range(0, len(no_chunk), batch_size):
        batch = no_chunk[start:start+batch_size]
        payload = {"layer_name": layer_name, "chunk_id": None, "items": batch}
        text = json.dumps(payload, ensure_ascii=False, indent=2)
        if len(text) > MAX_CHARS_PER_UNIT:
            text = text[:MAX_CHARS_PER_UNIT] + "\n...[unit truncated]"
        units.append({"unit_id": f"{layer_name}__batch_{start//batch_size:04d}", "chunk_id": None, "text": text})

    if not units:
        payload = {"layer_name": layer_name, "state": compact_obj(state, max_chars=MAX_CHARS_PER_UNIT)}
        text = json.dumps(payload, ensure_ascii=False, indent=2)
        units = [{"unit_id": f"{layer_name}__state", "chunk_id": None, "text": text[:MAX_CHARS_PER_UNIT]}]

    if MAX_UNITS_PER_LAYER is not None:
        units = units[:MAX_UNITS_PER_LAYER]
    return units

# Preview one checkpoint.
_preview_state = load_json(states_df.iloc[0]["state_path"])
_preview_units = build_evidence_units(_preview_state, states_df.iloc[0]["layer_name"])
print("Preview layer:", states_df.iloc[0]["layer_name"])
print("Preview units:", len(_preview_units))
print(_preview_units[0]["text"][:1200])

Preview layer: layer00_preprocessing
Preview units: 226
{
  "layer_name": "layer00_preprocessing",
  "chunk_id": "table_p0003_8_1",
  "items": [
    {
      "path": "root.document.chunks[0]",
      "chunk_id": "table_p0003_8_1",
      "item": {
        "chunk_id": "table_p0003_8_1",
        "text": "Alarme n°: 1001 texte: URGENCE ACTIVE type: Standard: Optionnel: effet: Alarme avec arrêt immédiate et contrôlée, ouverture de l’autorisa- tion hardware à l’alimentateur des axes/broches et CNC en urgence cause: Signale que le bouton coup-de-poing à été enfoncé intervention: Relâcher le bouton et appuyer sur la touche de démarrage de la machine. Voir la page 5 du schéma électrique, entrée X8.4 chargé de l’intervention: opérateur/chargé de l’entretien"
      }
    },
    {
      "path": "root.document.table_chunks[0]",
      "chunk_id": "table_p0003_8_1",
      "item": {
        "chunk_id": "table_p0003_8_1",
        "text": "Alarme n°: 1001 texte: URGENCE ACTIVE type: Standard: Optionnel: e

In [11]:

# ============================================================
# 8. LLM finalizer utilities
# ============================================================
def safe_json_from_text(text):
    if text is None:
        raise ValueError("empty response")
    if not isinstance(text, str):
        text = str(text)
    raw = text.strip()
    if not raw:
        raise ValueError("empty response")
    # Strip markdown fences.
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw).strip()
    # Direct parse.
    try:
        return json.loads(raw)
    except Exception:
        pass
    # Extract outermost JSON object or list.
    candidates = []
    for open_ch, close_ch in [("{", "}"), ("[", "]")]:
        a = raw.find(open_ch)
        b = raw.rfind(close_ch)
        if a >= 0 and b > a:
            candidates.append(raw[a:b+1])
    for cand in candidates:
        try:
            return json.loads(cand)
        except Exception:
            continue
    # Try a simple truncation repair.
    cand = raw
    if cand.count("{") > cand.count("}"):
        cand += "}" * (cand.count("{") - cand.count("}"))
    if cand.count("[") > cand.count("]"):
        cand += "]" * (cand.count("[") - cand.count("]"))
    try:
        return json.loads(cand)
    except Exception as e:
        raise ValueError(f"Could not parse JSON: {raw[:500]}") from e


def extract_message_content(response):
    # LiteLLM usually returns an object with choices[0].message.content.
    try:
        return response.choices[0].message.content
    except Exception:
        pass
    try:
        return response["choices"][0]["message"]["content"]
    except Exception:
        return str(response)


def finalizer_prompt(unit_text):
    return f"""
You are evaluating a NeoOLAF checkpoint for the XQuality Machine32 alarm/message document.
Your task is to generate final extraction outputs from the given checkpoint evidence only.

Use exactly these relation semantics:
- TRIGGERS: cause/explanation field -> alarm/message label.
- CAUSES: alarm/message label -> operational effect/consequence field.
- REQUIRES: alarm/message label -> intervention/action field.
- HANDLED_BY: alarm/message label -> responsible actor field.
- REFERENCES: alarm/message label -> technical reference/page/input/schema field.

Rules:
- Do not use gold annotations.
- Do not invent triples not supported by the checkpoint evidence.
- Accept French/English evidence.
- Return only valid JSON.
- Predicate must be one of: {", ".join(ALLOWED_PREDICATES)}.
- Prefer concise labels, but preserve technical meaning.

Return this exact JSON shape:
{{
  "triples": [
    {{"subject_label": "...", "predicate": "TRIGGERS|CAUSES|REQUIRES|HANDLED_BY|REFERENCES", "object_label": "...", "confidence": 0.0}}
  ],
  "kg": {{"nodes": [{{"id": "...", "label": "...", "type": "..."}}], "edges": [{{"source": "...", "predicate": "...", "target": "..."}}]}},
  "ontology": {{"classes": [], "object_properties": [], "axioms": []}}
}}

Checkpoint evidence:
{unit_text}
""".strip()


def call_litellm(messages, model=MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    from litellm import completion
    return completion(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )


def finalize_unit_with_llm(unit):
    prompt = finalizer_prompt(unit["text"])
    messages = [
        {"role": "system", "content": "You are a strict information extraction finalizer. Return only JSON."},
        {"role": "user", "content": prompt},
    ]
    last_error = None
    started = time.time()
    for attempt in range(LLM_RETRIES + 1):
        try:
            response = call_litellm(messages)
            raw = extract_message_content(response)
            parsed = safe_json_from_text(raw)
            return {
                "unit_id": unit["unit_id"],
                "chunk_id": unit.get("chunk_id"),
                "ok": True,
                "attempt": attempt,
                "elapsed_seconds": time.time() - started,
                "raw_response": raw,
                "parsed": parsed,
                "error": None,
            }
        except Exception as e:
            last_error = repr(e)
            if attempt < LLM_RETRIES:
                time.sleep(RETRY_SLEEP_SECONDS * (attempt + 1))
    return {
        "unit_id": unit["unit_id"],
        "chunk_id": unit.get("chunk_id"),
        "ok": False,
        "attempt": LLM_RETRIES,
        "elapsed_seconds": time.time() - started,
        "raw_response": None,
        "parsed": None,
        "error": last_error,
    }


def normalize_finalizer_output(parsed):
    if isinstance(parsed, list):
        # Some models return a list of objects. Merge triples.
        merged = {"triples": [], "kg": {"nodes": [], "edges": []}, "ontology": {"classes": [], "object_properties": [], "axioms": []}}
        for item in parsed:
            norm = normalize_finalizer_output(item)
            merged["triples"].extend(norm.get("triples", []))
            merged["kg"]["nodes"].extend(norm.get("kg", {}).get("nodes", []))
            merged["kg"]["edges"].extend(norm.get("kg", {}).get("edges", []))
            merged["ontology"]["classes"].extend(norm.get("ontology", {}).get("classes", []))
            merged["ontology"]["object_properties"].extend(norm.get("ontology", {}).get("object_properties", []))
            merged["ontology"]["axioms"].extend(norm.get("ontology", {}).get("axioms", []))
        return merged
    if not isinstance(parsed, dict):
        return {"triples": [], "kg": {"nodes": [], "edges": []}, "ontology": {"classes": [], "object_properties": [], "axioms": []}}
    triples = parsed.get("triples") or parsed.get("relations") or parsed.get("edges") or []
    kg = parsed.get("kg") or {"nodes": parsed.get("nodes", []), "edges": parsed.get("edges", [])}
    ontology = parsed.get("ontology") or {"classes": parsed.get("classes", []), "object_properties": parsed.get("object_properties", []), "axioms": parsed.get("axioms", [])}
    return {"triples": triples, "kg": kg if isinstance(kg, dict) else {}, "ontology": ontology if isinstance(ontology, dict) else {}}


def triples_from_finalizer_outputs(unit_results):
    triples = []
    kg_nodes = []
    kg_edges = []
    ont_classes = []
    ont_props = []
    ont_axioms = []
    for r in unit_results:
        if not r.get("ok"):
            continue
        norm = normalize_finalizer_output(r.get("parsed"))
        for t in norm.get("triples", []) or []:
            if isinstance(t, dict):
                triples.append(t)
        kg = norm.get("kg", {}) or {}
        if isinstance(kg, dict):
            kg_nodes.extend(kg.get("nodes") or [])
            kg_edges.extend(kg.get("edges") or [])
        ont = norm.get("ontology", {}) or {}
        if isinstance(ont, dict):
            ont_classes.extend(ont.get("classes") or [])
            ont_props.extend(ont.get("object_properties") or ont.get("properties") or [])
            ont_axioms.extend(ont.get("axioms") or [])
    triples = _dedup_triples(triples)
    return {
        "triples": triples,
        "kg": {"nodes": kg_nodes, "edges": kg_edges},
        "ontology": {"classes": ont_classes, "object_properties": ont_props, "axioms": ont_axioms},
    }

print("LLM finalizer utilities loaded.")

LLM finalizer utilities loaded.


In [12]:

# ============================================================
# 9. Run one layer checkpoint
# ============================================================
def slugify(x):
    x = str(x)
    x = re.sub(r"[^a-zA-Z0-9._-]+", "_", x).strip("_")
    return x[:120] or "checkpoint"


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def append_jsonl(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def run_layer_checkpoint(row):
    state_path = Path(row["state_path"])
    layer_name = str(row["layer_name"])
    layer_slug = f"{int(row.get('layer_num', 999)):02d}_{slugify(layer_name)}_{hashlib.md5(str(state_path).encode()).hexdigest()[:8]}"
    layer_output_dir = OUTPUT_ROOT / layer_slug
    layer_output_dir.mkdir(parents=True, exist_ok=True)

    final_output_path = layer_output_dir / "final_output.json"
    metrics_path = layer_output_dir / "metrics.json"
    unit_outputs_path = layer_output_dir / "unit_outputs.jsonl"

    if final_output_path.exists() and metrics_path.exists() and not FORCE_RERUN:
        final_output = load_json(final_output_path)
        metrics = load_json(metrics_path)
        return {
            "layer_name": layer_name,
            "state_path": str(state_path),
            "output_dir": str(layer_output_dir),
            "status": "cached",
            **metrics.get("flat", {}),
        }

    if unit_outputs_path.exists() and FORCE_RERUN:
        unit_outputs_path.unlink()

    t0 = time.time()
    state = load_json(state_path)
    units = build_evidence_units(state, layer_name)

    print(f"\n=== Layer: {layer_name} ===")
    print(f"State: {state_path}")
    print(f"Units to process: {len(units)}")
    print(f"Output dir: {layer_output_dir}")
    print(f"Model: {MODEL}; workers={MAX_WORKERS}; max_tokens={MAX_TOKENS}")

    unit_results = []
    errors = []

    iterator_desc = f"{layer_name} LLM calls"
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(finalize_unit_with_llm, unit): idx for idx, unit in enumerate(units)}
        future_iter = as_completed(futures)
        if USE_TQDM and tqdm is not None:
            future_iter = tqdm(future_iter, total=len(futures), desc=iterator_desc)
        completed = 0
        for future in future_iter:
            idx = futures[future]
            try:
                result = future.result()
            except Exception as e:
                result = {"unit_id": units[idx].get("unit_id"), "chunk_id": units[idx].get("chunk_id"), "ok": False, "error": repr(e), "traceback": traceback.format_exc()}
            unit_results.append(result)
            append_jsonl(unit_outputs_path, result)
            completed += 1
            if not result.get("ok"):
                errors.append(result)
                print(f"[ERROR] {layer_name} unit {idx}: {result.get('error')}")
            if VERBOSE and (completed % PROGRESS_EVERY_N_CALLS == 0 or completed == len(units)):
                ok_count = sum(1 for r in unit_results if r.get("ok"))
                print(f"[{layer_name}] completed {completed}/{len(units)} calls | ok={ok_count} errors={len(errors)}")

    final_output = triples_from_finalizer_outputs(unit_results)
    final_output["metadata"] = {
        "layer_name": layer_name,
        "state_path": str(state_path),
        "model": MODEL,
        "units": len(units),
        "ok_units": sum(1 for r in unit_results if r.get("ok")),
        "failed_units": sum(1 for r in unit_results if not r.get("ok")),
        "elapsed_seconds": time.time() - t0,
    }

    metrics = evaluate_triples(final_output["triples"], gold_triples)
    flat = {
        "layer_name": layer_name,
        "state_path": str(state_path),
        "output_dir": str(layer_output_dir),
        "units": len(units),
        "ok_units": final_output["metadata"]["ok_units"],
        "failed_units": final_output["metadata"]["failed_units"],
        "elapsed_seconds": final_output["metadata"]["elapsed_seconds"],
        "pred_triples": metrics["counts"]["pred_triples"],
        "relation_precision": metrics["relation"]["precision"],
        "relation_recall": metrics["relation"]["recall"],
        "relation_f1": metrics["relation"]["f1"],
        "entity_precision": metrics["entity"]["precision"],
        "entity_recall": metrics["entity"]["recall"],
        "entity_f1": metrics["entity"]["f1"],
    }
    metrics["flat"] = flat

    write_json(final_output_path, final_output)
    write_json(metrics_path, metrics)
    pd.DataFrame(final_output["triples"]).to_csv(layer_output_dir / "predicted_triples.csv", index=False)
    pd.DataFrame([
        {"predicate": k, **v} for k, v in metrics["per_relation"].items()
    ]).to_csv(layer_output_dir / "per_relation_metrics.csv", index=False)

    print(f"Finished {layer_name}: triples={flat['pred_triples']} relation_f1={flat['relation_f1']:.3f} elapsed={flat['elapsed_seconds']:.1f}s")
    return {"status": "done", **flat}

print("Layer runner ready.")

Layer runner ready.


In [ ]:

# ============================================================
# 10. Run checkpoint-to-output ablation over all discovered layers
# ============================================================
status_csv = OUTPUT_ROOT / "run_status.csv"
progress_log = OUTPUT_ROOT / "progress.log"

all_rows = states_df.to_dict("records")
print("Total checkpoints to run:", len(all_rows))
print("Output root:", OUTPUT_ROOT)

summary_rows = []
start_all = time.time()

if PARALLELIZE_LAYERS:
    print("PARALLELIZE_LAYERS=True. Running layers in parallel.")
    with ThreadPoolExecutor(max_workers=LAYER_MAX_WORKERS) as executor:
        futures = {executor.submit(run_layer_checkpoint, row): i for i, row in enumerate(all_rows)}
        layer_iter = as_completed(futures)
        if USE_TQDM and tqdm is not None:
            layer_iter = tqdm(layer_iter, total=len(futures), desc="Checkpoint layers")
        for future in layer_iter:
            i = futures[future]
            try:
                result = future.result()
            except Exception as e:
                result = {"layer_name": all_rows[i].get("layer_name"), "state_path": str(all_rows[i].get("state_path")), "status": "failed", "error": repr(e)}
                print("[LAYER ERROR]", result)
            summary_rows.append(result)
            pd.DataFrame(summary_rows).to_csv(status_csv, index=False)
            with open(progress_log, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")
else:
    layer_iter = enumerate(all_rows, start=1)
    if USE_TQDM and tqdm is not None:
        layer_iter = tqdm(list(layer_iter), desc="Checkpoint layers")
    for layer_idx, row in layer_iter:
        print(f"\n[{layer_idx}/{len(all_rows)}] Starting {row.get('layer_name')}")
        try:
            result = run_layer_checkpoint(row)
        except Exception as e:
            result = {"layer_name": row.get("layer_name"), "state_path": str(row.get("state_path")), "status": "failed", "error": repr(e), "traceback": traceback.format_exc()}
            print("[LAYER ERROR]", result["error"])
        summary_rows.append(result)
        pd.DataFrame(summary_rows).to_csv(status_csv, index=False)
        with open(progress_log, "a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
        print(f"[{layer_idx}/{len(all_rows)}] Finished {row.get('layer_name')} | status={result.get('status')}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_ROOT / "checkpoint_to_output_summary.csv", index=False)
print("\nAll done in", time.time() - start_all, "seconds")
display(summary_df)

Total checkpoints to run: 15
Output root: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\checkpoint_to_output_llm_ablation


Checkpoint layers:   0%|          | 0/15 [00:00<?, ?it/s]


[1/15] Starting layer00_preprocessing

=== Layer: layer00_preprocessing ===
State: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\layer01_only\layer00_preprocessing\state.json
Units to process: 226
Output dir: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\checkpoint_to_output_llm_ablation\00_layer00_preprocessing_dfe81338
Model: openrouter/openai/gpt-oss-20b; workers=4; max_tokens=1800


20:58:25 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
20:58:25 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


[layer00_preprocessing] completed 10/226 calls | ok=10 errors=0


...

[ERROR] layer00_preprocessing unit 45: ValueError('Could not parse JSON: {\n  "triples": [\n    {\n      "subject_label": "Le programme de gestion des outils jumelés a trouvé que le nombre de pièces de consigne pour un ou plusieurs outils est terminé.",\n      "predicate": "TRIGGERS",\n      "object_label": "SISTER TOOL: STOP AT END OF CYCLE",\n      "confidence": 0.0\n    },\n    {\n      "subject_label": "SISTER TOOL: STOP AT END OF CYCLE",\n      "predicate": "CAUSES",\n      "object_label": "À la fin du cycle (M01), arrête l’exécution automatique du programme.",\n      "conf')


[layer00_preprocessing] completed 50/226 calls | ok=46 errors=4


[layer00_preprocessing] completed 60/226 calls | ok=56 errors=4


[ERROR] layer00_preprocessing unit 57: ValueError('Could not parse JSON: {\n  "triples": [\n    {\n      "subject_label": "Malgré les tentatives effectuées en mode automatique pour déblo- quer le convoyeur ou la vis sans fin du coincement, le problème persiste",\n      "predicate": "TRIGGERS",\n      "object_label": "CONVOYEUR DES COPEAUX BLOQUÉ",\n      "confidence": 1.0\n    },\n    {\n      "subject_label": "CONVOYEUR DES COPEAUX BLOQUÉ",\n      "predicate": "CAUSES",\n      "object_label": "Seule visualisation du message",\n      "confidence": 1.0\n    },\n    {\n      "subject')


[layer00_preprocessing] completed 70/226 calls | ok=65 errors=5


In [ ]:

# ============================================================
# 11. Evaluate native NeoOLAF final states for comparison
# ============================================================
native_rows = []
for _, row in states_df.iterrows():
    try:
        state = load_json(row["state_path"])
        triples = extract_native_triples_from_state(state)
        if not triples:
            continue
        metrics = evaluate_triples(triples, gold_triples)
        native_rows.append({
            "variant": "native_neoolaf_state",
            "layer_name": row["layer_name"],
            "state_path": str(row["state_path"]),
            "pred_triples": metrics["counts"]["pred_triples"],
            "relation_precision": metrics["relation"]["precision"],
            "relation_recall": metrics["relation"]["recall"],
            "relation_f1": metrics["relation"]["f1"],
            "entity_precision": metrics["entity"]["precision"],
            "entity_recall": metrics["entity"]["recall"],
            "entity_f1": metrics["entity"]["f1"],
        })
    except Exception as e:
        print("Native eval failed for", row["state_path"], repr(e))

native_df = pd.DataFrame(native_rows)
native_df.to_csv(OUTPUT_ROOT / "native_neoolaf_state_metrics.csv", index=False)
print("Native states with extractable triples:", len(native_df))
if not native_df.empty:
    display(native_df.sort_values("relation_f1", ascending=False).head(20))

In [ ]:

# ============================================================
# 12. Consolidated comparison tables
# ============================================================
summary_path = OUTPUT_ROOT / "checkpoint_to_output_summary.csv"
summary_df = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame(summary_rows)

llm_df = summary_df.copy()
if not llm_df.empty:
    llm_df["variant"] = "checkpoint_to_output_llm"
    llm_df["display_name"] = llm_df["layer_name"].astype(str) + " + LLM finalizer"

native_comp = native_df.copy() if 'native_df' in globals() else pd.DataFrame()
if not native_comp.empty:
    native_comp["display_name"] = native_comp["layer_name"].astype(str) + " native NeoOLAF"

common_cols = [
    "variant", "display_name", "layer_name", "state_path", "pred_triples",
    "relation_precision", "relation_recall", "relation_f1",
    "entity_precision", "entity_recall", "entity_f1"
]
comparison = pd.concat([
    llm_df[[c for c in common_cols if c in llm_df.columns]],
    native_comp[[c for c in common_cols if c in native_comp.columns]],
], ignore_index=True)

comparison.to_csv(OUTPUT_ROOT / "full_comparison_metrics.csv", index=False)
print("Saved:", OUTPUT_ROOT / "full_comparison_metrics.csv")
display(comparison.sort_values("relation_f1", ascending=False).head(50))

In [ ]:

# ============================================================
# 13. Plots
# ============================================================
if plt is None:
    print("matplotlib not available")
else:
    plot_df = summary_df.copy()
    if "layer_num" not in plot_df.columns:
        # Recover layer numbers by joining with states_df.
        plot_df = plot_df.merge(states_df[["state_path", "layer_num"]].astype({"state_path": str}), on="state_path", how="left")
    plot_df = plot_df.sort_values(["layer_num", "layer_name"])

    if not plot_df.empty:
        plt.figure(figsize=(12, 5))
        plt.plot(range(len(plot_df)), plot_df["relation_precision"], marker="o", label="Relation precision")
        plt.plot(range(len(plot_df)), plot_df["relation_recall"], marker="o", label="Relation recall")
        plt.plot(range(len(plot_df)), plot_df["relation_f1"], marker="o", label="Relation F1")
        plt.xticks(range(len(plot_df)), plot_df["layer_name"].astype(str), rotation=75, ha="right")
        plt.ylim(0, 1.05)
        plt.ylabel("Score")
        plt.title("Checkpoint-to-output LLM ablation — relation metrics")
        plt.legend()
        plt.tight_layout()
        fig_path = OUTPUT_ROOT / "relation_metrics_by_checkpoint.png"
        plt.savefig(fig_path, dpi=180)
        plt.show()
        print("Saved:", fig_path)

        plt.figure(figsize=(12, 5))
        plt.plot(range(len(plot_df)), plot_df["entity_precision"], marker="o", label="Entity precision")
        plt.plot(range(len(plot_df)), plot_df["entity_recall"], marker="o", label="Entity recall")
        plt.plot(range(len(plot_df)), plot_df["entity_f1"], marker="o", label="Entity F1")
        plt.xticks(range(len(plot_df)), plot_df["layer_name"].astype(str), rotation=75, ha="right")
        plt.ylim(0, 1.05)
        plt.ylabel("Score")
        plt.title("Checkpoint-to-output LLM ablation — entity metrics")
        plt.legend()
        plt.tight_layout()
        fig_path = OUTPUT_ROOT / "entity_metrics_by_checkpoint.png"
        plt.savefig(fig_path, dpi=180)
        plt.show()
        print("Saved:", fig_path)

In [ ]:

# ============================================================
# 14. Final summary markdown for paper/report
# ============================================================
best_llm = None
if not summary_df.empty and "relation_f1" in summary_df.columns:
    best_llm = summary_df.sort_values("relation_f1", ascending=False).iloc[0].to_dict()

best_native = None
if 'native_df' in globals() and not native_df.empty:
    best_native = native_df.sort_values("relation_f1", ascending=False).iloc[0].to_dict()

report_lines = []
report_lines.append("# Checkpoint-to-output LLM ablation summary\n")
report_lines.append(f"Model: `{MODEL}`  ")
report_lines.append(f"Checkpoints evaluated: `{len(summary_df)}`  ")
report_lines.append(f"Output directory: `{OUTPUT_ROOT}`  \n")
if best_llm:
    report_lines.append("## Best checkpoint + LLM finalizer\n")
    report_lines.append(f"- Layer: `{best_llm.get('layer_name')}`")
    report_lines.append(f"- Predicted triples: `{best_llm.get('pred_triples')}`")
    report_lines.append(f"- Relation P/R/F1: `{best_llm.get('relation_precision'):.3f}` / `{best_llm.get('relation_recall'):.3f}` / `{best_llm.get('relation_f1'):.3f}`")
    report_lines.append(f"- Entity P/R/F1: `{best_llm.get('entity_precision'):.3f}` / `{best_llm.get('entity_recall'):.3f}` / `{best_llm.get('entity_f1'):.3f}`\n")
if best_native:
    report_lines.append("## Best native NeoOLAF state\n")
    report_lines.append(f"- Layer: `{best_native.get('layer_name')}`")
    report_lines.append(f"- Predicted triples: `{best_native.get('pred_triples')}`")
    report_lines.append(f"- Relation P/R/F1: `{best_native.get('relation_precision'):.3f}` / `{best_native.get('relation_recall'):.3f}` / `{best_native.get('relation_f1'):.3f}`")
    report_lines.append(f"- Entity P/R/F1: `{best_native.get('entity_precision'):.3f}` / `{best_native.get('entity_recall'):.3f}` / `{best_native.get('entity_f1'):.3f}`\n")

report_text = "\n".join(report_lines)
(OUTPUT_ROOT / "checkpoint_to_output_ablation_summary.md").write_text(report_text, encoding="utf-8")
print(report_text)